# Horse Racing RL — Phase 3: Multi-Agent Self-Play

Train archetypes to compete against each other using iterative self-play.

**Prerequisites:** Upload 4 Phase 2 archetype ONNX models.

**Approach:** Each archetype trains against frozen ONNX opponents (the other 3 archetypes).
After one round, the updated models become the next round's opponents. 3 rounds total.

**Variable field size:** Each episode randomly samples 3-19 opponents for generalization.

**Outputs:** 4 competition-ready ONNX models.

## 1. Setup

In [ ]:
import os

if os.path.exists("hr-simulation"):
    !cd hr-simulation && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git

%cd hr-simulation

In [ ]:
!pip uninstall -y gym 2>/dev/null
!pip install -q stable-baselines3 gymnasium pettingzoo numpy torch onnx onnxruntime

In [ ]:
from horse_racing.engine import HorseRacingEngine, EngineConfig
from horse_racing.types import HorseAction, HORSE_COUNT, MAX_HORSE_COUNT
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.self_play_env import SelfPlayEnv
from horse_racing.reward import compute_reward, ARCHETYPES
print(f"Engine OK — supports up to {MAX_HORSE_COUNT} horses")
print(f"Archetypes: {ARCHETYPES}")
print(f"SelfPlayEnv loaded")

## 2. Upload Phase 2 ONNX Models

Upload the 4 archetype models from Phase 2:
- `jockey_front_runner.onnx`
- `jockey_stalker.onnx`
- `jockey_closer.onnx`
- `jockey_presser.onnx`

In [ ]:
import onnx
import onnx.numpy_helper
import torch
import numpy as np
from pathlib import Path
from stable_baselines3 import PPO
import onnxruntime as ort

# Upload ONNX files
ONNX_DIR = Path("phase2_models")
ONNX_DIR.mkdir(exist_ok=True)

expected_files = [f"jockey_{a}.onnx" for a in ARCHETYPES]
missing = [f for f in expected_files if not (ONNX_DIR / f).exists()]

if missing:
    try:
        from google.colab import files
        print(f"Upload {len(missing)} ONNX files: {missing}")
        uploaded = files.upload()
        for name, data in uploaded.items():
            (ONNX_DIR / name).write_bytes(data)
    except ImportError:
        raise FileNotFoundError(f"Missing: {missing}. Place them in {ONNX_DIR}/")

# Build opponent path map
opponent_paths = {}
for arch in ARCHETYPES:
    path = ONNX_DIR / f"jockey_{arch}.onnx"
    assert path.exists(), f"Missing: {path}"
    opponent_paths[arch] = str(path)
    sess = ort.InferenceSession(str(path))
    test = np.zeros((1, 26), dtype=np.float32)
    out = sess.run(["action"], {"obs": test})[0][0]
    print(f"  {arch:15s} → [{out[0]:.4f}, {out[1]:.4f}] ✓")

# Detect architecture from first model
onnx_model = onnx.load(str(ONNX_DIR / f"jockey_{ARCHETYPES[0]}.onnx"))
onnx_weights = {}
for init in onnx_model.graph.initializer:
    onnx_weights[init.name] = torch.tensor(np.array(onnx.numpy_helper.to_array(init)))

layer0_out = onnx_weights["mlp_extractor.policy_net.0.weight"].shape[0]
layer1_out = onnx_weights["mlp_extractor.policy_net.2.weight"].shape[0]
print(f"\nArchitecture: 26 → {layer0_out} → {layer1_out} → 2")

## 3. ONNX ↔ SB3 Utilities

In [ ]:
import torch.nn as nn


class PolicyNetwork(nn.Module):
    """Wraps SB3 policy for ONNX export."""
    def __init__(self, sb3_policy):
        super().__init__()
        self.features_extractor = sb3_policy.features_extractor
        self.mlp_extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, obs):
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


def onnx_to_state_dict(onnx_path):
    """Load ONNX weights into a dict compatible with SB3 policy."""
    model = onnx.load(onnx_path)
    weights = {}
    for init in model.graph.initializer:
        weights[init.name] = torch.tensor(np.array(onnx.numpy_helper.to_array(init)))
    return weights


def export_to_onnx(sb3_model, output_path):
    """Export SB3 model to ONNX."""
    wrapper = PolicyNetwork(sb3_model.policy)
    wrapper.eval()
    dummy = torch.zeros(1, 26, dtype=torch.float32)
    torch.onnx.export(
        wrapper, dummy, output_path,
        input_names=["obs"], output_names=["action"],
        dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
        opset_version=17, dynamo=False,
    )


def load_weights_into_model(model, onnx_path):
    """Inject ONNX weights into an SB3 model."""
    weights = onnx_to_state_dict(onnx_path)
    sd = model.policy.state_dict()
    for name in weights:
        if name in sd:
            sd[name] = weights[name]
    model.policy.load_state_dict(sd)


print("Utilities ready")

## 4. Training Configuration

In [ ]:
DEVICE = "cpu"
N_ENVS = 32               # scaled up from 8 — Colab Pro has ~50GB RAM
BATCH_SIZE = 1024          # scaled up from 256
NUM_ROUNDS = 3
TIMESTEPS_PER_ARCHETYPE = 300_000

# Per-round hyperparameter schedules
LR_SCHEDULE = [3e-4, 1e-4, 5e-5]
ENT_SCHEDULE = [0.01, 0.005, 0.003]

TRAINING_TRACKS = [
    "tracks/curriculum_2_gentle_oval.json",
    "tracks/curriculum_3_tight_oval.json",
    "tracks/tokyo.json",
    "tracks/kokura.json",
    "tracks/hanshin.json",
    "tracks/kyoto.json",
    "tracks/tokyo_2600.json",
]

MIN_OPPONENTS = 3
MAX_OPPONENTS = 7  # keep manageable for training speed

total_steps = NUM_ROUNDS * len(ARCHETYPES) * TIMESTEPS_PER_ARCHETYPE
print(f"Rounds: {NUM_ROUNDS}")
print(f"Archetypes: {ARCHETYPES}")
print(f"Steps/archetype/round: {TIMESTEPS_PER_ARCHETYPE:,}")
print(f"Total steps: {total_steps:,}")
print(f"Opponents per episode: {MIN_OPPONENTS}-{MAX_OPPONENTS}")
print(f"Tracks: {len(TRAINING_TRACKS)}")

## 5. Training Callback

In [ ]:
import time
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback


class SelfPlayCallback(BaseCallback):
    def __init__(self, stage_name, total_timesteps, global_step_offset, global_total,
                 print_freq=2048):
        super().__init__()
        self.stage_name = stage_name
        self.total = total_timesteps
        self.print_freq = print_freq
        self.global_offset = global_step_offset
        self.global_total = global_total
        self.start_time = None

    def _on_training_start(self):
        self.start_time = time.time()

    def _on_step(self) -> bool:
        if self.n_calls % self.print_freq == 0:
            elapsed = time.time() - self.start_time
            steps = self.num_timesteps
            sps = steps / elapsed if elapsed > 0 else 0
            pct = 100 * steps / self.total
            overall = 100 * (self.global_offset + steps) / self.global_total
            eta = (self.total - steps) / sps if sps > 0 else 0

            if len(self.model.ep_info_buffer) > 0:
                mean_r = sum(ep['r'] for ep in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)
                reward_str = f"reward: {mean_r:8.2f}"
            else:
                reward_str = "reward:      n/a"

            print(f"  [{self.stage_name}] {pct:5.1f}% | overall: {overall:4.1f}% | "
                  f"steps: {steps:>8,} | {reward_str} | {sps:.0f} sps | ETA: {eta/60:.1f}m")
        return True


print("Callback ready")

## 6. Train — Iterative Self-Play

In [ ]:
import random

checkpoint_dir = Path("checkpoints/phase3")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

VecEnvClass = SubprocVecEnv if N_ENVS > 1 else DummyVecEnv

# Current opponent ONNX paths — starts with Phase 2 models
current_opponents = dict(opponent_paths)  # copy

total_start = time.time()
global_steps_done = 0

for rnd in range(NUM_ROUNDS):
    lr = LR_SCHEDULE[rnd]
    ent = ENT_SCHEDULE[rnd]
    round_dir = checkpoint_dir / f"round_{rnd}"
    round_dir.mkdir(exist_ok=True)

    print(f"\n{'#'*60}")
    print(f"Round {rnd} — LR={lr}, entropy={ent}")
    print(f"Opponents from: {'Phase 2' if rnd == 0 else f'Round {rnd-1}'}")
    print(f"{'#'*60}")

    round_models = {}

    for arch_idx, archetype in enumerate(ARCHETYPES):
        print(f"\n{'='*60}")
        print(f"[R{rnd} {arch_idx+1}/{len(ARCHETYPES)}] Training: {archetype} — {TIMESTEPS_PER_ARCHETYPE:,} steps")
        print(f"{'='*60}")

        # Build opponent list: all archetypes except the trainee
        opp_paths = [current_opponents[a] for a in ARCHETYPES if a != archetype]

        def make_env(arch=archetype, opps=opp_paths):
            return SelfPlayEnv(
                tracks=TRAINING_TRACKS,
                max_steps=5000,
                opponent_onnx_paths=opps,
                trainee_archetype=arch,
                min_opponents=MIN_OPPONENTS,
                max_opponents=MAX_OPPONENTS,
            )

        env = VecEnvClass([make_env for _ in range(N_ENVS)])

        model = PPO(
            "MlpPolicy", env, verbose=0,
            policy_kwargs={"net_arch": [layer0_out, layer1_out]},
            n_steps=2048, batch_size=BATCH_SIZE, n_epochs=10,
            learning_rate=lr, gamma=0.99, gae_lambda=0.95,
            clip_range=0.2, vf_coef=0.5, ent_coef=ent,
            device=DEVICE,
        )

        # Initialize weights
        if rnd == 0:
            load_weights_into_model(model, opponent_paths[archetype])
            print(f"  Init from Phase 2 ONNX")
        else:
            prev_zip = checkpoint_dir / f"round_{rnd-1}" / f"jockey_{archetype}.zip"
            prev_model = PPO.load(str(prev_zip), env=env, device=DEVICE)
            model.policy.load_state_dict(prev_model.policy.state_dict())
            print(f"  Init from Round {rnd-1} checkpoint")

        callback = SelfPlayCallback(
            f"R{rnd}-{archetype}", TIMESTEPS_PER_ARCHETYPE,
            global_steps_done, total_steps,
        )
        model.learn(total_timesteps=TIMESTEPS_PER_ARCHETYPE, callback=callback)

        # Save SB3 checkpoint + ONNX
        sb3_path = round_dir / f"jockey_{archetype}"
        model.save(str(sb3_path))
        onnx_path = round_dir / f"jockey_{archetype}.onnx"
        export_to_onnx(model, str(onnx_path))
        round_models[archetype] = model
        print(f"  Saved → {sb3_path} + {onnx_path}")

        env.close()
        global_steps_done += TIMESTEPS_PER_ARCHETYPE

    # Update opponents for next round
    for arch in ARCHETYPES:
        current_opponents[arch] = str(round_dir / f"jockey_{arch}.onnx")

total_time = time.time() - total_start
print(f"\nAll rounds complete in {total_time/60:.1f} minutes")

## 7. Head-to-Head Evaluation

In [ ]:
# Load final round models
final_round = NUM_ROUNDS - 1
final_dir = checkpoint_dir / f"round_{final_round}"

sessions = {}
for arch in ARCHETYPES:
    onnx_path = str(final_dir / f"jockey_{arch}.onnx")
    sessions[arch] = ort.InferenceSession(onnx_path)

eval_tracks = ["tracks/tokyo.json", "tracks/hanshin.json", "tracks/kokura.json"]

for track in eval_tracks:
    engine = HorseRacingEngine(track)
    track_name = Path(track).stem

    for tick in range(5000):
        all_obs = engine.get_observations()
        actions = []
        for i, arch in enumerate(ARCHETYPES):
            arr = engine.obs_to_array(all_obs[i]).reshape(1, -1)
            action = sessions[arch].run(["action"], {"obs": arr})[0][0]
            actions.append(HorseAction(float(action[0]), float(action[1])))
        engine.step(actions)

        # Check if all finished
        if all(all_obs[i]["finished"] for i in range(4)):
            break

    placements = engine.get_placements()
    final_obs = engine.get_observations()
    print(f"\n{track_name.upper()} ({tick+1} ticks):")
    for i, arch in enumerate(ARCHETYPES):
        o = final_obs[i]
        status = "FINISHED" if o["finished"] else f"{o['track_progress']:.1%}"
        print(f"  P{placements[i]}: {arch:15s} | {status} | stamina: {o['stamina_ratio']:.3f}")

## 8. Download Final Models

In [ ]:
try:
    from google.colab import files
    for arch in ARCHETYPES:
        onnx_path = str(final_dir / f"jockey_{arch}.onnx")
        files.download(onnx_path)
    print("Downloads started!")
except ImportError:
    print(f"Not in Colab. Models at: {final_dir}/jockey_*.onnx")